In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Given a 2D grid of 32-bit floating point values, apply one iteration of the 5-point Jacobi stencil:
  each interior cell of the output is set to the average of its four cardinal neighbors (top, bottom,
  left, right) from the input grid. Boundary cells (first/last row and column) are copied unchanged
  from the input to the output.
</p>

<svg width="320" height="280" viewBox="0 0 320 280" xmlns="http://www.w3.org/2000/svg"
     style="display:block; margin:20px auto;" font-family="monospace" font-size="11">
  <rect width="320" height="280" rx="8" fill="#222"/>
  <defs>
    <marker id="arrj" markerWidth="7" markerHeight="5" refX="7" refY="2.5" orient="auto">
      <polygon points="0 0, 7 2.5, 0 5" fill="#44aa66"/>
    </marker>
  </defs>

  <!-- Title -->
  <text x="160" y="18" text-anchor="middle" fill="#ccc" font-size="11">5-Point Jacobi Stencil</text>

  <!-- 5x5 grid, cell 44x34, origin (30, 28) -->
  <!-- Row 0 (boundary) -->
  <rect x="30"  y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="126" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="174" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="222" y="28" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 1 -->
  <rect x="30"  y="66" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="66" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="126" y="66" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="174" y="66" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="222" y="66" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 2 -->
  <rect x="30"  y="104" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="104" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="126" y="104" width="44" height="34" rx="2" fill="#1e2d4d" stroke="#4477bb" stroke-width="2"/>
  <rect x="174" y="104" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="222" y="104" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 3 -->
  <rect x="30"  y="142" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="142" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="126" y="142" width="44" height="34" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="2"/>
  <rect x="174" y="142" width="44" height="34" rx="2" fill="#333" stroke="#555"/>
  <rect x="222" y="142" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <!-- Row 4 (boundary) -->
  <rect x="30"  y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="78"  y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="126" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="174" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>
  <rect x="222" y="180" width="44" height="34" rx="2" fill="#2a2a2a" stroke="#444"/>

  <!-- Neighbor labels -->
  <text x="148" y="88" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">T</text>
  <text x="100" y="126" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">L</text>
  <text x="148" y="124" text-anchor="middle" fill="#4477bb" font-size="9">(2,2)</text>
  <text x="196" y="126" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">R</text>
  <text x="148" y="164" text-anchor="middle" font-weight="bold" fill="#44aa66" font-size="12">B</text>

  <!-- Arrows from neighbors to center -->
  <line x1="148" y1="100" x2="148" y2="107" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="122" y1="121" x2="129" y2="121" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="174" y1="121" x2="167" y2="121" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>
  <line x1="148" y1="142" x2="148" y2="135" stroke="#44aa66" stroke-width="1.5" marker-end="url(#arrj)"/>

  <!-- Legend -->
  <rect x="30" y="228" width="14" height="14" rx="2" fill="#1e2d4d" stroke="#4477bb" stroke-width="1.5"/>
  <text x="48" y="240" fill="#ccc" font-size="9">Center cell</text>
  <rect x="115" y="228" width="14" height="14" rx="2" fill="#1e3d2d" stroke="#44aa66" stroke-width="1.5"/>
  <text x="133" y="240" fill="#ccc" font-size="9">Neighbors</text>
  <rect x="200" y="228" width="14" height="14" rx="2" fill="#2a2a2a" stroke="#444" stroke-width="1.5"/>
  <text x="218" y="240" fill="#ccc" font-size="9">Boundary</text>

  <!-- Formula -->
  <text x="160" y="264" text-anchor="middle" fill="#ccc" font-size="11">output[i,j] = &#xbc; &#xd7; (top + bottom + left + right)</text>
</svg>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>output</code></li>
  <li>Read exclusively from <code>input</code> and write exclusively to <code>output</code> (do not update <code>input</code>)</li>
</ul>

<h2>Example:</h2>
<p>
Input ($4 \times 4$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0 \\
9.0 & 10.0 & 11.0 & 12.0 \\
13.0 & 14.0 & 15.0 & 16.0
\end{bmatrix}
$$
Output ($4 \times 4$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 & 4.0 \\
5.0 & 6.0 & 7.0 & 8.0 \\
9.0 & 10.0 & 11.0 & 12.0 \\
13.0 & 14.0 & 15.0 & 16.0
\end{bmatrix}
$$
Interior cell $(1,1)$: $0.25 \times (\text{input}[0,1] + \text{input}[2,1] + \text{input}[1,0] + \text{input}[1,2])$
$= 0.25 \times (2.0 + 10.0 + 5.0 + 7.0) = 6.0$<br>
Interior cell $(1,2)$: $0.25 \times (\text{input}[0,2] + \text{input}[2,2] + \text{input}[1,1] + \text{input}[1,3])$
$= 0.25 \times (3.0 + 11.0 + 6.0 + 8.0) = 7.0$<br>
Interior cell $(2,1)$: $0.25 \times (\text{input}[1,1] + \text{input}[3,1] + \text{input}[2,0] + \text{input}[2,2])$
$= 0.25 \times (6.0 + 14.0 + 9.0 + 11.0) = 10.0$<br>
Interior cell $(2,2)$: $0.25 \times (\text{input}[1,2] + \text{input}[3,2] + \text{input}[2,1] + \text{input}[2,3])$
$= 0.25 \times (7.0 + 15.0 + 10.0 + 12.0) = 11.0$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>rows</code>, <code>cols</code> &le; 16,384</li>
  <li>Input values are in the range [-100, 100]</li>
  <li>All values are 32-bit floats</li>
  <li>Performance is measured with <code>rows</code> = 8,192, <code>cols</code> = 8,192</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// input, output are device pointers
extern "C" void solve(const float* input, float* output, int rows, int cols) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, rows: cute.Int32, cols: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# input is a tensor on the GPU
@jax.jit
def solve(input: jax.Array, rows: int, cols: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    rows: Int32,
    cols: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, rows: int, cols: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, rows: int, cols: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/69_jacobi_stencil_2d/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
